# Blood Cell Subtype Classification — Corrected Version

This notebook keeps the same project goal: classify **Eosinophil, Lymphocyte, Monocyte, and Neutrophil**.

### Main corrections
1. Use `TRAIN` for training.
2. Use the dataset's `TEST` folder as validation during model development.
3. Keep `TEST_SIMPLE` untouched as the final test set when it is available.
4. Use stronger but medically reasonable augmentation to improve generalization.
5. Fine-tune only the upper DenseNet layers with a small learning rate.
6. Add test-time augmentation (TTA) for final prediction.
7. Report confusion matrix, classification report, standard accuracy, and TTA accuracy.

> Do not train on `TEST_SIMPLE` if you want an honest final accuracy.


In [ ]:
import os
import zipfile
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from google.colab import files
from PIL import Image

from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Dataset loading from Google Drive

from google.colab import drive
drive.mount("/content/drive")

# ZIP file location in your Google Drive
ZIP_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset2-master.zip"

assert os.path.exists(ZIP_PATH), (
    f"Zip not found at {ZIP_PATH} - check the path/file name"
)

print(
    "Found:",
    ZIP_PATH,
    "-",
    round(os.path.getsize(ZIP_PATH) / 1e6, 1),
    "MB"
)

EXTRACT_PATH = "/content/blood_cell_dataset"

if os.path.exists(EXTRACT_PATH):
    shutil.rmtree(EXTRACT_PATH)

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted successfully from Google Drive.")


In [ ]:
DATASET_ROOT = None

for root, dirs, files_list in os.walk(EXTRACT_PATH):
    if "TRAIN" in dirs and "TEST" in dirs:
        DATASET_ROOT = root
        break

if DATASET_ROOT is None:
    raise FileNotFoundError("Could not find a folder containing TRAIN and TEST.")

TRAIN_DIR = os.path.join(DATASET_ROOT, "TRAIN")
TEST_DIR = os.path.join(DATASET_ROOT, "TEST")
TEST_SIMPLE_DIR = os.path.join(DATASET_ROOT, "TEST_SIMPLE")

print("Dataset root:", DATASET_ROOT)
print("TRAIN:", TRAIN_DIR)
print("TEST :", TEST_DIR)
print("TEST_SIMPLE:", TEST_SIMPLE_DIR if os.path.exists(TEST_SIMPLE_DIR) else "Not found")

In [ ]:
CLASS_NAMES = [
    "EOSINOPHIL",
    "LYMPHOCYTE",
    "MONOCYTE",
    "NEUTROPHIL"
]

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
NUM_CLASSES = len(CLASS_NAMES)

image_extensions = (".jpg", ".jpeg", ".png", ".bmp")

def count_images(folder):
    result = {}
    for class_name in CLASS_NAMES:
        class_dir = os.path.join(folder, class_name)
        result[class_name] = sum(
            1 for f in os.listdir(class_dir)
            if f.lower().endswith(image_extensions)
        )
    return result

for name, folder in [
    ("TRAIN", TRAIN_DIR),
    ("TEST", TEST_DIR)
]:
    counts = count_images(folder)
    print(name, counts, "TOTAL =", sum(counts.values()))

if os.path.exists(TEST_SIMPLE_DIR):
    counts = count_images(TEST_SIMPLE_DIR)
    print("TEST_SIMPLE", counts, "TOTAL =", sum(counts.values()))

## 1. Data pipeline

The original notebook created validation data by randomly taking 15% from `TRAIN`. That makes validation very similar to training, while the later evaluation was performed on a different augmented split. Here, `TEST` is used as the development/validation set and `TEST_SIMPLE` is reserved for final testing when available.


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    class_names=CLASS_NAMES,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    class_names=CLASS_NAMES,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

if os.path.exists(TEST_SIMPLE_DIR):
    final_test_ds = tf.keras.utils.image_dataset_from_directory(
        TEST_SIMPLE_DIR,
        class_names=CLASS_NAMES,
        label_mode="categorical",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False
    )
    FINAL_TEST_NAME = "TEST_SIMPLE"
else:
    final_test_ds = val_ds
    FINAL_TEST_NAME = "TEST"

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
final_test_ds = final_test_ds.prefetch(AUTOTUNE)

print("Final untouched test set:", FINAL_TEST_NAME)

In [ ]:
plt.figure(figsize=(12, 8))

for images, labels in train_ds.take(1):
    for i in range(min(12, len(images))):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(CLASS_NAMES[np.argmax(labels[i].numpy())])
        plt.axis("off")

plt.tight_layout()
plt.show()

## 2. Data augmentation

The purpose is to reduce dependence on the exact appearance of the training images and improve generalization to differently augmented or original images.


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.10)
], name="blood_cell_augmentation")

## 3. DenseNet121 transfer-learning model

ImageNet weights provide useful low-level visual features. The classifier head is trained first, then only the upper part of DenseNet is fine-tuned with a small learning rate.


In [ ]:
base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.25)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

HEAD_PATH = "/content/densenet_head_best.keras"

head_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        HEAD_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=head_callbacks
)

In [ ]:
model = tf.keras.models.load_model(HEAD_PATH)

base_model = model.get_layer("densenet121")
base_model.trainable = True

# Freeze most of DenseNet and fine-tune only the upper layers.
fine_tune_from = max(0, len(base_model.layers) - 100)

for i, layer in enumerate(base_model.layers):
    if i < fine_tune_from:
        layer.trainable = False
    elif isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

print("Total DenseNet layers:", len(base_model.layers))
print("Fine-tuning last:", len(base_model.layers) - fine_tune_from, "layers")

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

FINETUNE_PATH = "/content/densenet_finetuned_best.keras"

finetune_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        FINETUNE_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=finetune_callbacks
)

In [ ]:
head_best = max(history_head.history["val_accuracy"])
fine_best = max(history_finetune.history["val_accuracy"])

if fine_best >= head_best:
    FINAL_MODEL_PATH = FINETUNE_PATH
else:
    FINAL_MODEL_PATH = HEAD_PATH

final_model = tf.keras.models.load_model(FINAL_MODEL_PATH)

print("Best head validation accuracy :", round(head_best * 100, 2), "%")
print("Best fine-tune validation accuracy:", round(fine_best * 100, 2), "%")
print("Selected model:", FINAL_MODEL_PATH)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history_head.history["accuracy"], label="Head Train")
plt.plot(history_head.history["val_accuracy"], label="Head Validation")

offset = len(history_head.history["accuracy"])

plt.plot(
    range(offset, offset + len(history_finetune.history["accuracy"])),
    history_finetune.history["accuracy"],
    label="Fine-tune Train"
)

plt.plot(
    range(offset, offset + len(history_finetune.history["val_accuracy"])),
    history_finetune.history["val_accuracy"],
    label="Fine-tune Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 4. Standard final evaluation

This is the ordinary `model.evaluate()` result on the untouched final test set.


In [ ]:
test_loss, test_accuracy = final_model.evaluate(
    final_test_ds,
    verbose=1
)

print("=" * 60)
print("FINAL TEST RESULT")
print("=" * 60)
print("Test set:", FINAL_TEST_NAME)
print("Accuracy:", round(test_accuracy * 100, 2), "%")
print("Loss:", round(test_loss, 4))

In [ ]:
y_true = []
y_pred = []

for images, labels in final_test_ds:
    probabilities = final_model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(probabilities, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4
))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
)

fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, values_format="d", cmap="Blues")
plt.title("Final Test Confusion Matrix")
plt.xticks(rotation=45)
plt.show()

print("PER-CLASS ACCURACY")
print("=" * 40)

for i, class_name in enumerate(CLASS_NAMES):
    total = np.sum(cm[i])
    correct = cm[i, i]
    print(class_name, ":", round(correct / total * 100, 2), "%")

## 5. Test-Time Augmentation (TTA)

TTA makes several prediction views of the same image and averages the probabilities. It does not use the test labels and can improve robustness to orientation.


In [ ]:
def predict_tta_batch(model, images):
    views = [
        images,
        tf.image.flip_left_right(images),
        tf.image.flip_up_down(images),
        tf.image.rot90(images, 1)
    ]

    probabilities = []

    for view in views:
        probabilities.append(model(view, training=False))

    return tf.reduce_mean(tf.stack(probabilities, axis=0), axis=0)

In [ ]:
tta_true = []
tta_pred = []

for images, labels in final_test_ds:
    probabilities = predict_tta_batch(final_model, images)
    tta_true.extend(np.argmax(labels.numpy(), axis=1))
    tta_pred.extend(np.argmax(probabilities.numpy(), axis=1))

tta_true = np.array(tta_true)
tta_pred = np.array(tta_pred)

tta_accuracy = np.mean(tta_true == tta_pred)

print("=" * 60)
print("TTA FINAL TEST RESULT")
print("=" * 60)
print("Test set:", FINAL_TEST_NAME)
print("Standard accuracy:", round(test_accuracy * 100, 2), "%")
print("TTA accuracy     :", round(tta_accuracy * 100, 2), "%")

## 6. Save the final model and class names


In [ ]:
FINAL_SAVE_PATH = "/content/blood_cell_subtype_final_corrected.keras"
CLASS_NAMES_PATH = "/content/class_names.json"

final_model.save(FINAL_SAVE_PATH)

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(CLASS_NAMES, f)

print("Model saved:", FINAL_SAVE_PATH)
print("Classes saved:", CLASS_NAMES_PATH)

## 7. Single-image prediction

The same preprocessing is used automatically because `preprocess_input` is part of the model. TTA is used for the displayed prediction.


In [ ]:
uploaded = files.upload()

for filename in uploaded.keys():
    image = Image.open(filename).convert("RGB")
    original = image.copy()

    resized = image.resize(IMG_SIZE)
    image_array = np.array(resized, dtype=np.float32)
    image_array = np.expand_dims(image_array, axis=0)

    probabilities = predict_tta_batch(final_model, image_array).numpy()[0]

    predicted_index = int(np.argmax(probabilities))
    predicted_class = CLASS_NAMES[predicted_index]
    confidence = probabilities[predicted_index] * 100

    print("=" * 60)
    print("BLOOD CELL SUBTYPE PREDICTION")
    print("=" * 60)
    print("Image      :", filename)
    print("Subtype    :", predicted_class)
    print("Confidence :", round(confidence, 2), "%")

    print("\nCLASS PROBABILITIES")
    for class_name, probability in zip(CLASS_NAMES, probabilities):
        print(f"{class_name:12s}: {probability * 100:.2f}%")

    plt.figure(figsize=(6, 6))
    plt.imshow(original)
    plt.title(f"{predicted_class} - {confidence:.2f}%")
    plt.axis("off")
    plt.show()